In [ ]:
%gui qt

import mne

mne.viz.set_browser_backend("qt")

import numpy as np
import pandas as pd

In [ ]:
#%pip install mne numpy pandas statsmodels

import os, json, shutil, sys, math, warnings
from pathlib import Path
import numpy as np, pandas as pd
import mne
from mne.preprocessing import ICA, annotate_muscle_zscore
import statsmodels.api as sm

warnings.filterwarnings("ignore", category=FutureWarning)
print("MNE:", mne.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)


## 1. CONFIG

In [ ]:
import mne
PROJECT_DIR = Path(r"E:\Develop\MEG\...")
DERIV_DIR   = PROJECT_DIR / "derivatives"
RAW_DIR     = PROJECT_DIR
DERIV_DIR.mkdir(parents=True, exist_ok=True)

SESSION_LABEL = "C"
ENV_LABEL     = "stable"

_ALL_STIM_SETS = [
    [149, 151, 153],
    [150, 152, 154],
]
RESP_CODES_BY_SESSION = {
    "S": {},
    "C": {},
}

def _detect_stim_codes(run_files, stim_ch="STI101", candidate_sets=None):
    if not run_files:
        return sorted(set(c for s in candidate_sets for c in s))
    try:
        raw = mne.io.read_raw_fif(run_files[0], preload=False, verbose=False)
        ev = mne.find_events(raw, stim_channel=stim_ch, shortest_event=1,
                             initial_event=False, verbose=False)
        present = set(ev[:, 2].tolist())
        best_set, best_n = None, 0
        for s in candidate_sets:
            n = len(present & set(s))
            if n > best_n:
                best_n, best_set = n, s
        if best_set and best_n > 0:
            print(f"[auto-detect] stim codes detected: {best_set} ({best_n} events)")
            return best_set
    except Exception as e:
        print(f"[auto-detect] stim code detection failed ({e}), using all candidates")
    fallback = sorted(set(c for s in candidate_sets for c in s))
    print(f"[auto-detect] fallback stim codes: {fallback}")
    return fallback

CALIBRATION_FILE = None
CROSSTALK_FILE   = None

ER_RAW_PATH   = str(PROJECT_DIR / "empty_room.fif")

ER_DIR    = DERIV_DIR / f"{SESSION_LABEL}_empty_room"; ER_DIR.mkdir(exist_ok=True)

SESSION_PROFILE_JSON = DERIV_DIR / f"{SESSION_LABEL}_session_profile.json"
EVENTS_JSON          = DERIV_DIR / "events_auto.json"
STATE_JSON           = DERIV_DIR / "pipeline_state.json"

RUN_FILES = sorted([
    p for p in RAW_DIR.glob("*.fif")
    if all(x not in p.name.lower() for x in ("rest", "empty"))
])

if not RUN_FILES:
    existing_run_dirs = sorted([
        d for d in DERIV_DIR.glob("run*") if d.is_dir()
        and any(d.glob("raw_clean.fif"))
    ], key=lambda d: int(d.name.replace("run", "")))
    N_RUNS = len(existing_run_dirs)
    print(f"WARNING: No raw .fif files found in RAW_DIR. "
          f"Inferred N_RUNS={N_RUNS} from derivatives (re-epoch only mode).")
else:
    N_RUNS = min(6, len(RUN_FILES))

_detected_stim = _detect_stim_codes(list(RUN_FILES), stim_ch="STI101", candidate_sets=_ALL_STIM_SETS)
STIM_CODES_BY_SESSION = {SESSION_LABEL: _detected_stim}


In [ ]:
MANUAL_QC = {
    "pre_sss":   True,
    "post_filt": True,
    "ica":       True,
    "events":    True,
    "epochs":    True,
}
QC_PRE_SSS = True
QC_POST_FILT = True
QC_ICA = True
QC_EPOCHS = True
QC_PSD = False

ER_MANUAL_QC = True
ALIGN_EMPTY_ROOM_TO_RUN1 = True

INCLUDE_EOG_ECG_IN_EPOCHS = False
OVERWRITE_EVENT_MAP = False
ICA_MANUAL_EXCLUDE = []


In [ ]:
STIM_CH = "STI101"
EOG_CHS = ["EOG061","EOG062"]
ECG_CHS = ["ECG063"]

In [ ]:
FEEDBACK_CODES = {
    "fb_win"      : 190,  # correct & reward
    "fb_corr_trap": 191,  # correct, trap -> no reward
    "fb_loss"     : 192,  # incorrect & no reward
    "fb_trap_win" : 193,  # incorrect, trap -> reward
    "fb_norp"     : 204,  # no response
    "fb_norp2"    : 205,
}

FB_EVENT_IDS = [190, 191, 192, 193]


In [ ]:
NOTCH = [50., 100., 150.]
HP, LP = 1.0, 40.0
REJECT = dict(grad=4000e-13, mag=4e-12, eog=250e-6)

LOCK = "feedback"
TMIN, TMAX = -0.2, 0.8
BASELINE = (-0.2, 0.0)

RT_WINDOW = (0.0, 1.0)

DO_SSS = True
DO_TSSS = False  # if no tSSS, keep False
ST_DURATION = 10.0
DEST_JSON = DERIV_DIR / "destination_dev_head_t.json"

ROI_WINDOWS = {
    "early": (0.12, 0.30),
    "late":  (0.30, 0.60),
}

KNOWN_BADS = ["MEG2642","MEG2613", "MEG2633", "MEG2111"]
FORCE_REDO = False


## 2. Helper functions


In [ ]:
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import mne


def log(*args):
    print(">>", *args)


def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)


def save_dev_head_t(raw: mne.io.BaseRaw, path: Path):
    dht = raw.info.get("dev_head_t", None)
    if dht is None:
        raise RuntimeError("dev_head_t not found in raw.info")

    from_ = getattr(dht, "from_", None) or (dht.get("from") if isinstance(dht, dict) else None)
    to    = getattr(dht, "to",    None) or (dht.get("to")   if isinstance(dht, dict) else None)
    trans = getattr(dht, "trans", None) or (dht.get("trans")if isinstance(dht, dict) else None)
    if trans is None:
        raise RuntimeError("dev_head_t found, but field 'trans' is missing")

    data = dict(from_=int(from_), to=int(to), trans=np.asarray(trans).tolist())
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)


def load_dev_head_t(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        d = json.load(f)
    return {"from": int(d["from_"]), "to": int(d["to"]), "trans": np.array(d["trans"])}


def events_by_codes(raw: mne.io.BaseRaw, stim_channel: str, valid_codes):
    if isinstance(valid_codes, dict):
        codes = list(valid_codes.values())
    else:
        codes = list(valid_codes)

    ev = mne.find_events(raw, stim_channel=stim_channel, shortest_event=1, initial_event=False)
    return ev[np.isin(ev[:, 2], codes)]


def build_metadata_fb(raw: mne.io.BaseRaw,
                      ev_stim: np.ndarray,
                      ev_fb:   np.ndarray,
                      ev_resp: np.ndarray,
                      rt_window=(0.0, 1.0)) -> pd.DataFrame:
    sf = float(raw.info['sfreq'])
    lo, hi = rt_window

    ev_stim = ev_stim[np.argsort(ev_stim[:, 0])]
    ev_fb   = ev_fb[np.argsort(ev_fb[:, 0])]
    ev_fb   = ev_fb[ev_fb[:, 2] != 205]
    ev_resp = ev_resp[np.argsort(ev_resp[:, 0])]

    rows = []
    n_no_fb = 0

    for s_samp, _, s_code in ev_stim:
        s_samp = int(s_samp)

        next_stim_mask = ev_stim[:, 0] > s_samp
        next_stim_samp = int(ev_stim[next_stim_mask, 0][0]) if next_stim_mask.any() else int(s_samp + 10 * sf)

        fb_candidates = ev_fb[(ev_fb[:, 0] > s_samp) & (ev_fb[:, 0] < next_stim_samp)]

        if len(fb_candidates) == 0:
            n_no_fb += 1
            f_code, f_samp = -1, s_samp
            outcome, correct, reward = "no_fb", -1, -1
            stim_to_fb = np.nan
        else:
            f_samp, _, f_code = fb_candidates[0]
            f_samp, f_code = int(f_samp), int(f_code)
            if   f_code == 190: outcome, correct, reward = "win",        1, 1
            elif f_code == 191: outcome, correct, reward = "corr_trap",  1, 0
            elif f_code == 192: outcome, correct, reward = "loss",       0, 0
            elif f_code == 193: outcome, correct, reward = "trap_win",   0, 1
            else:               outcome, correct, reward = "other",     -1,-1
            stim_to_fb = float((f_samp - s_samp) / sf)

        rt_end = min(f_samp if f_code != -1 else s_samp + int(hi * sf),
                     int(s_samp + hi * sf))
        resp_in = ev_resp[(ev_resp[:, 0] >= s_samp) & (ev_resp[:, 0] <= rt_end)]
        if resp_in.size:
            r_samp, _, r_code = resp_in[0]
            rt = float((int(r_samp) - s_samp) / sf)
            r_code = int(r_code)
        else:
            r_code, rt = -1, np.nan

        rows.append(dict(
            onset_s      = float(s_samp / sf),
            stim_code    = int(s_code),
            fb_code      = f_code,
            resp_code    = r_code,
            RT_s         = rt,
            outcome      = outcome,
            correct      = int(correct),
            reward       = int(reward),
            stim_to_fb_s = stim_to_fb,
        ))

    if n_no_fb:
        log(f"build_metadata_fb: {n_no_fb} stim(s) without feedback (outcome='no_fb')")

    return pd.DataFrame(rows)


def build_metadata_trialwise(raw: mne.io.BaseRaw,
                             ev_stim: np.ndarray,
                             ev_fb:   np.ndarray,
                             ev_resp: np.ndarray,
                             rt_window=(0.0, 1.0),
                             fb_allow=(190,191,192,193),
                             report_dupes=True) -> pd.DataFrame:

    sf = float(raw.info['sfreq'])
    lo, hi = rt_window

    ev_stim = ev_stim[np.argsort(ev_stim[:,0])]
    ev_fb   = ev_fb[np.argsort(ev_fb[:,0])]
    ev_resp = ev_resp[np.argsort(ev_resp[:,0])]

    if fb_allow:
        ev_fb = ev_fb[np.isin(ev_fb[:, 2], list(fb_allow))]

    rows, n_skipped = [], 0
    seen_stims = {}

    for f_samp, _, f_code in ev_fb:
        idx = np.where(ev_stim[:, 0] <= f_samp)[0]
        if idx.size == 0:
            n_skipped += 1
            continue
        s_samp, _, s_code = ev_stim[idx[-1]]
        s_key = int(s_samp)

        if report_dupes and s_key in seen_stims:
            log(f"build_metadata_trialwise: duplicate FB for stim at sample {s_key}")
        seen_stims[s_key] = seen_stims.get(s_key, 0) + 1

        rt_end = min(int(f_samp), int(s_samp + hi * sf))
        resp_in = ev_resp[(ev_resp[:, 0] >= s_samp) & (ev_resp[:, 0] <= rt_end)]

        if resp_in.size:
            r_samp, _, r_code = resp_in[0]
            rt = (r_samp - s_samp) / sf
            r_code = int(r_code)
        else:
            r_code, rt = -1, np.nan

        if   f_code == 190: outcome, correct, reward = "win",        1, 1
        elif f_code == 191: outcome, correct, reward = "corr_trap",  1, 0
        elif f_code == 192: outcome, correct, reward = "loss",       0, 0
        elif f_code == 193: outcome, correct, reward = "trap_win",   0, 1
        else:               outcome, correct, reward = "other",     -1,-1

        rows.append(dict(
            onset_s      = float(f_samp / sf),
            stim_code    = int(s_code),
            fb_code      = int(f_code),
            resp_code    = int(r_code),
            RT_s         = (rt if np.isfinite(rt) else np.nan),
            outcome      = outcome,
            correct      = int(correct),
            reward       = int(reward),
            stim_to_fb_s = float((f_samp - s_samp) / sf),
        ))

    if n_skipped:
        log(f"build_metadata_trialwise: skipped FB without preceding stimulus: {n_skipped}")

    return pd.DataFrame(rows)


def safe_save_epochs(ep: mne.Epochs, path: Path):
    tmp = path.with_suffix(".tmp-epo.fif")
    ep.save(tmp, overwrite=True)
    if path.exists():
        path.unlink()
    tmp.rename(path)


def update_state(run_idx, status: str, extra: dict | None = None):
    state_path = Path(globals().get("STATE_JSON", DERIV_DIR / "pipeline_state.json"))

    state = {}
    if state_path.exists():
        try:
            state = json.loads(state_path.read_text(encoding="utf-8"))
        except Exception:
            state = {}

    state.setdefault("runs", {})
    payload = {"status": status, "updated_at": datetime.now().isoformat(timespec="seconds")}
    if extra:
        for k, v in extra.items():
            try:
                json.dumps(v)
                payload[k] = v
            except Exception:
                payload[k] = str(v)

    state["runs"][str(run_idx)] = payload
    state_path.write_text(json.dumps(state, indent=2), encoding="utf-8")
    return state


def update_session_profile(path: Path, updates: dict):
    data = {}
    if path.exists():
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            data = {}
    data.update(updates)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")
    return data


def _save_event_maps(path: Path, stim_dict, resp_dict, fb_event_ids):
    event_map = {
        "stim_codes":   stim_dict if isinstance(stim_dict, dict) else {str(c): int(c) for c in stim_dict},
        "resp_codes":   resp_dict if isinstance(resp_dict, dict) else {str(c): int(c) for c in resp_dict},
        "fb_event_ids": list(fb_event_ids),
    }
    path.write_text(json.dumps(event_map, indent=2), encoding="utf-8")


def get_destination_transform(dest_json: Path):
    if dest_json is None or not dest_json.exists():
        return None
    try:
        dest_dict = load_dev_head_t(dest_json)
        _T = np.array(dest_dict["trans"], dtype=float)
        from mne.transforms import Transform
        return Transform(dest_dict.get("from", "meg"),
                         dest_dict.get("to", "head"),
                         _T)
    except Exception as e:
        log(f"DEST_JSON invalid -> no align ({e})")
        return None


def maybe_compute_headpos_summary(raw: mne.io.BaseRaw, out_json: Path):
    try:
        chpi_amps = mne.chpi.compute_chpi_amplitudes(raw, verbose=False)
        chpi_locs = mne.chpi.compute_chpi_locs(raw.info, chpi_amps, verbose=False)
        head_pos = mne.chpi.compute_head_pos(raw.info, chpi_locs)
        if head_pos is None or len(head_pos) == 0:
            log("headpos: no cHPI data")
            return None
        trans = head_pos[:, 1:4]
        disp = np.linalg.norm(trans - trans[0], axis=1) * 1000.0
        summary = {
            "n_samples":        int(len(head_pos)),
            "movement_mm_p50":  float(np.percentile(disp, 50)),
            "movement_mm_p95":  float(np.percentile(disp, 95)),
            "movement_mm_max":  float(np.max(disp)),
        }
        out_json.write_text(json.dumps(summary, indent=2), encoding="utf-8")
        return summary
    except BaseException as e:
        log(f"headpos: skipped ({type(e).__name__}: {e})")
        return None


## 3. Empty-room (optional)


In [ ]:
from pathlib import Path
import json
import mne


def _apply_bads_from_profile(raw, session_label, run_dir=None):
    try:
        prof = json.loads(SESSION_PROFILE_JSON.read_text(encoding="utf-8"))
    except Exception:
        prof = {}

    session_bads_file = DERIV_DIR / f"bads_session_{session_label}.txt"
    run_bads_file = (run_dir / "bads.txt") if run_dir else None

    sources = {
        "raw.info": set(raw.info.get("bads", [])),
        "KNOWN_BADS": set(globals().get("KNOWN_BADS", []) or []),
        "SESSION_BADS_FILE": set(),
        "RUN_BADS_FILE": set(),
        "empty_room_profile": set(prof.get("bads_from_er", []) or []),
    }
    if session_bads_file.exists():
        sources["SESSION_BADS_FILE"] = {
            ln.strip() for ln in session_bads_file.read_text(encoding="utf-8").splitlines() if ln.strip()
        }
    if run_bads_file and run_bads_file.exists():
        sources["RUN_BADS_FILE"] = {
            ln.strip() for ln in run_bads_file.read_text(encoding="utf-8").splitlines() if ln.strip()
        }

    known = set().union(*sources.values())
    known_existing = sorted([ch for ch in known if ch in raw.ch_names])
    raw.info["bads"] = known_existing
    return sources, known_existing


def _maxwell_or_copy(raw, destination, st_duration, coord_frame="head"):
    if not DO_SSS:
        return raw.copy()
    return mne.preprocessing.maxwell_filter(
        raw,
        origin="auto",
        st_duration=st_duration,
        destination=destination,
        calibration=CALIBRATION_FILE,
        cross_talk=CROSSTALK_FILE,
        bad_condition="warning",
        regularize="in",
        coord_frame=coord_frame,
        verbose=True
    )


def _filter_raw(raw):
    if NOTCH:
        raw.notch_filter(NOTCH, picks="meg")
    raw.filter(HP, LP, picks="meg", phase="zero-double")
    return raw


def process_empty_room():
    if not ER_RAW_PATH:
        log("[empty-room] ER_RAW_PATH is None -> skip")
        return None

    er_path = Path(ER_RAW_PATH)
    if not er_path.exists():
        log(f"[empty-room] ER_RAW_PATH does not exist -> skip: {er_path}")
        return None

    er_sss_path = ER_DIR / "raw_er_sss.fif"
    if er_sss_path.exists() and not FORCE_REDO:
        log(f"[empty-room] {er_sss_path.name} already exists -> skip")
        return er_sss_path

    log("[empty-room] read and preprocess")
    er_raw = mne.io.read_raw_fif(er_path, preload=True)
    _apply_bads_from_profile(er_raw, SESSION_LABEL, run_dir=None)

    if ER_MANUAL_QC:
        er_raw.plot(block=True)

    dest = get_destination_transform(DEST_JSON) if ALIGN_EMPTY_ROOM_TO_RUN1 else None
    if ALIGN_EMPTY_ROOM_TO_RUN1 and dest is None:
        log("[empty-room] DEST_JSON not found -> no align")

    coord_frame = "head" if er_raw.info.get("dev_head_t") is not None else "meg"
    er_sss = _maxwell_or_copy(er_raw, destination=dest, st_duration=(ST_DURATION if DO_TSSS else None), coord_frame=coord_frame)

    er_pre_path = ER_DIR / "raw_er_pre.fif"
    er_raw.save(er_pre_path, overwrite=True)

    ann_musc, _ = annotate_muscle_zscore(
        er_sss, ch_type="mag", threshold=5.5,
        min_length_good=0.05,
        filter_freq=[110, 140]
    )
    er_sss.set_annotations(er_sss.annotations + ann_musc)
    er_sss.save(er_sss_path, overwrite=True)

    update_session_profile(SESSION_PROFILE_JSON, {
        "bads_from_er": list(er_raw.info.get("bads", [])),
        "empty_room_raw": str(er_pre_path),
        "empty_room_sss": str(er_sss_path),
    })
    log(f"[empty-room] saved: {er_sss_path.name}")
    return er_sss_path


# Run once if desired
_ = process_empty_room()


## 4A. Run all runs sequentially (A1-A7 + combine)


In [ ]:
import warnings
import numpy as np
import pandas as pd
import mne
from collections import Counter, defaultdict

# ── Headpos QC отключён навсегда в этой ячейке ──────────────────────────────
DO_HEADPOS_QC = False

START_RUN_IDX = 1
END_RUN_IDX = N_RUNS

if "process_empty_room" not in globals():
    def process_empty_room():
        print("process_empty_room not defined; run the empty-room cell or set ER_RAW_PATH=None to skip.")
        return None

def process_run(run_idx: int):
    run_idx = int(run_idx)
    if run_idx < 1 or run_idx > len(RUN_FILES):
        raise ValueError(f"RUN_IDX out of range: {run_idx}")
    RUN_IDX = run_idx
    RUN_FILE = RUN_FILES[RUN_IDX - 1]

    run_dir = DERIV_DIR / f"run{RUN_IDX:02d}"
    ensure_dir(run_dir)
    out = {
        "raw_pre":      run_dir / "raw_pre_sss.fif",
        "raw_sss":      run_dir / "raw_sss.fif",
        "raw_filt":     run_dir / "raw_filt.fif",
        "raw_annot":    run_dir / "raw_annot.fif",
        "raw_ica":      run_dir / "raw_ica.fif",
        "raw_clean":    run_dir / "raw_clean.fif",
        "epochs":       run_dir / "epochs-epo.fif",
        "metadata_csv": run_dir / "metadata.csv",
        "events_csv":   run_dir / "events_counts.csv",
    }
    import json

    qc_collector = {
        "epochs_qc": {},
        "ica_qc": {},
        "annotations_qc": {},
        "psd_qc": {},
        "general": {}
    }

    try:
        prof = json.loads(SESSION_PROFILE_JSON.read_text(encoding="utf-8"))
    except Exception:
        prof = {}

    SESSION_BADS_FILE = DERIV_DIR / f"bads_session_{SESSION_LABEL}.txt"
    RUN_BADS_FILE     = run_dir / "bads.txt"

    # Read RAW
    log(f"[run {RUN_IDX}] A1. Read RAW: {RUN_FILE}")
    raw = mne.io.read_raw_fif(RUN_FILE, preload=False)

    # DO_HEADPOS_QC is skipped — headpos stats not needed

    sources = {
        "raw.info":           set(raw.info.get("bads", [])),
        "KNOWN_BADS":         set(globals().get("KNOWN_BADS", []) or []),
        "SESSION_BADS_FILE":  set(),
        "RUN_BADS_FILE":      set(),
        "empty_room_profile": set(prof.get("bads_from_er", []) or []),
    }
    if SESSION_BADS_FILE.exists():
        sources["SESSION_BADS_FILE"] = {
            ln.strip() for ln in SESSION_BADS_FILE.read_text(encoding="utf-8").splitlines() if ln.strip()
        }
    if RUN_BADS_FILE.exists():
        sources["RUN_BADS_FILE"] = {
            ln.strip() for ln in RUN_BADS_FILE.read_text(encoding="utf-8").splitlines() if ln.strip()
        }

    known = set().union(*sources.values())
    known_existing = sorted([ch for ch in known if ch in raw.ch_names])
    unknown = sorted(known - set(known_existing))
    if unknown:
        log(f"[run {RUN_IDX}] ignoring non-existent channels: {', '.join(unknown)}")
    raw.info["bads"] = known_existing

    def _nz(x): return len([_ for _ in x if _])
    log(
        f"[run {RUN_IDX}] Bad channels: {', '.join(raw.info['bads']) or 'none'} "
        f"(raw.info={_nz(sources['raw.info'])}, KNOWN_BADS={_nz(sources['KNOWN_BADS'])}, "
        f"session_file={_nz(sources['SESSION_BADS_FILE'])}, run_file={_nz(sources['RUN_BADS_FILE'])}, "
        f"empty_room={_nz(sources['empty_room_profile'])})"
    )

    if globals().get("QC_PRE_SSS", False):
        raw.plot(block=True)

    RUN_BADS_FILE.write_text("\n".join(raw.info["bads"]), encoding="utf-8")
    sess_union = set()
    if SESSION_BADS_FILE.exists():
        sess_union |= {
            ln.strip() for ln in SESSION_BADS_FILE.read_text(encoding="utf-8").splitlines() if ln.strip()
        }
    sess_union |= set(raw.info["bads"])
    SESSION_BADS_FILE.write_text("\n".join(sorted(sess_union)), encoding="utf-8")

    raw.save(out["raw_pre"], overwrite=True)
    if DO_SSS and RUN_IDX == 1:
        log(f"[run {RUN_IDX}] Saving dev_head_t -> {DEST_JSON.name}")
        save_dev_head_t(raw, DEST_JSON)
    update_state(RUN_IDX, "A1_done", {"raw_pre": str(out["raw_pre"]), "bads": list(raw.info["bads"])})

    # Maxwell / SSS
    if DO_SSS:
        dest = None
        if RUN_IDX > 1 and DEST_JSON.exists():
            dest = get_destination_transform(DEST_JSON)
        log(f"[run {RUN_IDX}] A2. Maxwell/SSS; destination: {'align to run1' if dest is not None else 'None'}")
        raw_sss = mne.preprocessing.maxwell_filter(
            raw, origin="auto",
            st_duration=(ST_DURATION if DO_TSSS else None),
            destination=dest,
            calibration=CALIBRATION_FILE,
            cross_talk=CROSSTALK_FILE,
            bad_condition="warning",
            regularize="in",
            verbose=True
        )
    else:
        raw_sss = raw.copy()
    raw_sss.save(out["raw_sss"], overwrite=True)
    update_state(RUN_IDX, "A2_done", {"raw_sss": str(out["raw_sss"]), "sss": bool(DO_SSS), "tsss": bool(DO_TSSS)})

    # Filter
    log(f"[run {RUN_IDX}] A3. Filters: notch {NOTCH}, band {HP}-{LP} Hz")
    rf = raw_sss.copy()
    if NOTCH:
        rf.notch_filter(NOTCH, picks="meg")
    rf.filter(HP, LP, picks="meg", phase="zero-double")
    rf.save(out["raw_filt"], overwrite=True)
    update_state(RUN_IDX, "A3_done", {"raw_filt": str(out["raw_filt"])})
    rf = mne.io.read_raw_fif(out["raw_filt"], preload=True)

    # Muscle annotation
    ann_probe, scores = annotate_muscle_zscore(
        rf, ch_type="mag", threshold=5.5, min_length_good=0.05, filter_freq=[110, 140]
    )
    ann_probe = mne.Annotations(ann_probe.onset, ann_probe.duration, ["muscle_probe"] * len(ann_probe))
    ann_probe = ann_probe.copy()
    ann_probe._orig_time = rf.annotations.orig_time
    rf.set_annotations(rf.annotations + ann_probe)

    if globals().get("QC_POST_FILT", False):
        rf.plot(block=True)

    rf.save(out["raw_annot"], overwrite=True)
    update_state(RUN_IDX, "A4_done", {"raw_annot": str(out["raw_annot"])})

    # QC figure: PSD of filtered + annotated raw
    try:
        import matplotlib.pyplot as plt
        _fd = run_dir / "figures"; _fd.mkdir(exist_ok=True)
        _pf = rf.compute_psd(fmin=1, fmax=150, n_jobs=1, verbose='ERROR').plot(show=False)
        _pf.suptitle(f"Run {RUN_IDX}: PSD filtered (pre-ICA)", fontsize=10)
        _pf.savefig(str(_fd / "psd_post_filter.png"), dpi=150, bbox_inches='tight')
        plt.close(_pf)
        log(f"[run {RUN_IDX}] Saved: psd_post_filter.png")
    except Exception as _e:
        log(f"[run {RUN_IDX}] PSD pre-ICA figure skipped: {_e}")

    # ICA
    from mne.preprocessing import ICA
    log(f"[run {RUN_IDX}] A5. ICA")
    ica = ICA(n_components=35, max_iter=500, method="fastica", fit_params={"fun": "exp"}, random_state=42, verbose=True)
    picks = mne.pick_types(rf.info, meg=True, eog=False, ecg=False)
    ica.fit(rf, picks=picks, decim=4, reject_by_annotation=True)

    # Auto-detect ECG / EOG components
    auto_ecg, auto_eog = [], []
    ecg_scores_saved, eog_scores_saved = {}, {}
    ecg_scores_arr = None
    eog_scores_arr = None

    _ecg_chs = [ch for ch in globals().get("ECG_CHS", []) if ch in rf.ch_names]
    _eog_chs = [ch for ch in globals().get("EOG_CHS", []) if ch in rf.ch_names]


    # Manual override: list -> all runs; dict {run_idx: [...]} -> per-run
    _manual_raw = globals().get("ICA_MANUAL_EXCLUDE", [])
    manual_excl = list(_manual_raw.get(RUN_IDX, [])) if isinstance(_manual_raw, dict) else list(_manual_raw)

    ica.exclude = sorted(set(manual_excl))
    log(f"[run {RUN_IDX}] ICA exclude (auto_ecg={auto_ecg}, auto_eog={auto_eog}, manual={manual_excl}) -> {ica.exclude}")

    # QC figures: component topomaps, scores, overlay
    try:
        import matplotlib.pyplot as plt
        _fd = run_dir / "figures"; _fd.mkdir(exist_ok=True)

        _comp_figs = ica.plot_components(show=False)
        if not isinstance(_comp_figs, list): _comp_figs = [_comp_figs]
        for _fi, _f in enumerate(_comp_figs):
            _f.savefig(str(_fd / f"ica_components_{_fi:02d}.png"), dpi=150, bbox_inches='tight')
            plt.close(_f)

        if ecg_scores_arr is not None:
            _sf = ica.plot_scores(ecg_scores_arr, exclude=auto_ecg, show=False,
                                  title=f"Run {RUN_IDX}: ECG correlation scores")
            if not isinstance(_sf, list): _sf = [_sf]
            for _fi, _f in enumerate(_sf):
                _f.savefig(str(_fd / f"ica_scores_ecg_{_fi:02d}.png"), dpi=150, bbox_inches='tight')
                plt.close(_f)

        if eog_scores_arr is not None:
            _sf = ica.plot_scores(eog_scores_arr, exclude=auto_eog, show=False,
                                  title=f"Run {RUN_IDX}: EOG correlation scores")
            if not isinstance(_sf, list): _sf = [_sf]
            for _fi, _f in enumerate(_sf):
                _f.savefig(str(_fd / f"ica_scores_eog_{_fi:02d}.png"), dpi=150, bbox_inches='tight')
                plt.close(_f)

        _mag_picks = mne.pick_types(rf.info, meg='mag')[:1]
        if len(_mag_picks):
            _ov = ica.plot_overlay(rf.copy(), exclude=ica.exclude, picks=_mag_picks, show=False)
            _ov.savefig(str(_fd / "ica_overlay.png"), dpi=150, bbox_inches='tight')
            plt.close(_ov)

        log(f"[run {RUN_IDX}] Saved ICA figures -> {_fd.name}/")
    except Exception as _e:
        log(f"[run {RUN_IDX}] ICA figures skipped: {_e}")

    if globals().get("QC_ICA", False):
        ica.plot_sources(rf, block=True)

    raw_clean = ica.apply(rf.copy())
    raw_clean.save(out["raw_clean"], overwrite=True)

    qc_collector["ica_qc"] = {
        "n_components":      int(ica.n_components_),
        "n_excluded":        len(ica.exclude),
        "excluded_indices":  list(map(int, ica.exclude)),
        "auto_ecg_indices":  auto_ecg,
        "auto_eog_indices":  auto_eog,
        "manual_indices":    manual_excl,
        "ecg_scores":        ecg_scores_saved,
        "eog_scores":        eog_scores_saved,
        "method":            "fastica",
        "fit_params":        {"fun": "exp"},
    }
    update_state(RUN_IDX, "A5_done", {"raw_clean": str(out["raw_clean"]), "ica_exclude": list(ica.exclude)})

    # Events + metadata
    rc = mne.io.read_raw_fif(out["raw_clean"], preload=True)
    events_stim = events_by_codes(rc, STIM_CH, STIM_CODES_BY_SESSION[SESSION_LABEL])
    events_resp = (events_by_codes(rc, STIM_CH, list(RESP_CODES_BY_SESSION[SESSION_LABEL].values()))
                   if RESP_CODES_BY_SESSION[SESSION_LABEL] else np.empty((0, 3), dtype=int))
    events_fb = events_by_codes(rc, STIM_CH, FB_EVENT_IDS)
    ev_all = np.vstack([events_stim, events_resp, events_fb])
    ev_all = ev_all[np.argsort(ev_all[:, 0])]
    log(f"[run {RUN_IDX}] found STIM={len(events_stim)}, RESP={len(events_resp)}, FB={len(events_fb)}")

    metadata = build_metadata_fb(rc, events_stim, events_fb, events_resp, rt_window=RT_WINDOW)
    metadata.to_csv(out["metadata_csv"], index=False)

    ev_df = pd.DataFrame(np.asarray([[e[0], 0, 1] for e in ev_all], dtype=int),
                         columns=["sample", "unknown", "event_id"])
    ev_df.to_csv(out["events_csv"], index=False)

    try:
        ev_json_path = EVENTS_JSON
        if not ev_json_path.exists() and not OVERWRITE_EVENT_MAP:
            _save_event_maps(ev_json_path, STIM_CODES_BY_SESSION[SESSION_LABEL],
                             RESP_CODES_BY_SESSION[SESSION_LABEL], FB_EVENT_IDS)
    except Exception as e:
        log(f"event_map: skipped ({e})")
    log(f"[run {RUN_IDX}] Built metadata with {len(metadata)} rows")

    def _summarize(s):
        if not hasattr(s, "values") or len(s) == 0:
            return {}
        v = s.values
        return {"n": int(v.size), "mean": float(np.mean(v)), "median": float(np.median(v)),
                "p05": float(np.percentile(v, 5)), "p95": float(np.percentile(v, 95)),
                "min": float(np.min(v)), "max": float(np.max(v))}

    def _count_codes(arr):
        if len(arr) == 0: return {}
        vals, cnts = np.unique(arr[:, 2].astype(int), return_counts=True)
        return {int(v): int(c) for v, c in zip(vals, cnts)}

    qc = {
        "n_events_total":    int(len(ev_all)),
        "n_stim":            int(len(events_stim)),
        "n_fb":              int(len(events_fb)),
        "n_resp":            int(len(events_resp)),
        "stim_code_counts":  _count_codes(events_stim),
        "fb_code_counts":    _count_codes(events_fb),
        "resp_code_counts":  _count_codes(events_resp),
        "stim_to_fb_s":      _summarize(metadata.get("stim_to_fb_s")),
        "rt_s":              _summarize(metadata.get("RT_s")),
    }
    qc_path = run_dir / "event_qc.json"
    qc_path.write_text(json.dumps(qc, indent=2), encoding="utf-8")
    update_state(RUN_IDX, "A6_done",
                 {"metadata": str(out["metadata_csv"]), "events_csv": str(out["events_csv"]),
                  "event_qc": str(qc_path)})
    log(f"STIM={len(events_stim)}, FB={len(events_fb)}, RESP={len(events_resp)}, metadata={len(metadata)}")

    # Epoching
    log(f"[run {RUN_IDX}] A7. Epoching: lock={LOCK}, tmin={TMIN}, tmax={TMAX}")
    picks_ep = mne.pick_types(rc.info, meg=True,
                              eog=bool(globals().get("INCLUDE_EOG_ECG_IN_EPOCHS", False)),
                              ecg=bool(globals().get("INCLUDE_EOG_ECG_IN_EPOCHS", False)))
    _event_id = ({str(int(c)): int(c) for c in np.unique(events_stim[:, 2])}
                 if len(events_stim) > 0 else {"stim": 1})
    _raw_reject = globals().get("REJECT") or None
    if _raw_reject:
        _ch_types_in_picks = set(mne.channel_type(rc.info, p) for p in picks_ep)
        _reject = {k: v for k, v in _raw_reject.items() if k in _ch_types_in_picks} or None
    else:
        _reject = None
    epochs = mne.Epochs(
        rc, events_stim,
        event_id=_event_id,
        tmin=TMIN, tmax=TMAX,
        baseline=BASELINE,
        reject=_reject,
        metadata=metadata.reset_index(drop=True) if metadata is not None else None,
        picks=picks_ep,
        preload=True,
        reject_by_annotation=True,
        verbose=True,
    )
    if globals().get("QC_EPOCHS", False):
        epochs.plot(block=True)
    epochs.save(out["epochs"], overwrite=True)
    log(f"[run {RUN_IDX}] ✓ Epochs saved: {len(epochs)} kept / {len(events_stim)} trials -> {out['epochs'].name}")

    # Epochs QC metrics
    n_total_ev = len(events_stim)
    n_kept     = len(epochs)
    n_dropped  = n_total_ev - n_kept
    drop_reasons = {}
    for reason_list in epochs.drop_log:
        for r in reason_list:
            drop_reasons[r] = drop_reasons.get(r, 0) + 1

    peak_data = epochs.get_data(picks="meg")
    peak_mag  = float(np.median(np.max(np.abs(peak_data), axis=(1, 2)))) if n_kept > 0 else float("nan")
    gfp_mean  = float(np.mean(np.std(peak_data, axis=1))) if n_kept > 0 else float("nan")

    qc_collector["epochs_qc"] = {
        "n_events":               int(n_total_ev),
        "n_epochs_kept":          int(n_kept),
        "n_epochs_dropped":       int(n_dropped),
        "retention_rate":         float(n_kept / n_total_ev) if n_total_ev > 0 else 0.0,
        "drop_reasons":           drop_reasons,
        "tmin":                   float(TMIN),
        "tmax":                   float(TMAX),
        "lock":                   str(LOCK),
        "peak_amplitude_median_T": peak_mag,
        "gfp_mean":               gfp_mean,
    }
    log(f"[run {RUN_IDX}] Epochs QC: kept={n_kept}/{n_total_ev} ({100*n_kept/n_total_ev:.1f}%), "
        f"drop reasons: {drop_reasons}")
    update_state(RUN_IDX, "A7_done", {"epochs": str(out["epochs"]), "n_epochs": int(n_kept),
                                       "n_dropped": int(n_dropped), "drop_reasons": drop_reasons})

    # ---- A7 QC figures: drop log, GFP, topomap ------------------------------
    try:
        import matplotlib.pyplot as plt
        _fd = run_dir / "figures"; _fd.mkdir(exist_ok=True)
        if n_kept > 0:
            _dl = epochs.plot_drop_log(show=False)
            _dl.savefig(str(_fd / "epochs_drop_log.png"), dpi=150, bbox_inches='tight')
            plt.close(_dl)

            _evoked = epochs.average(picks="meg")
            _times  = _evoked.times
            _gfp    = np.std(_evoked.data, axis=0)

            _fig_gfp, _ax = plt.subplots(figsize=(10, 3))
            _ax.plot(_times * 1000, _gfp * 1e15, color='steelblue', lw=1.5)
            _ax.axvline(0, color='r', linestyle='--', alpha=0.7, label='event')
            for _wn, (_t0, _t1) in globals().get("ROI_WINDOWS", {}).items():
                _ax.axvspan(_t0 * 1000, _t1 * 1000, alpha=0.12, label=_wn)
            _ax.set_xlabel("Time (ms)")
            _ax.set_ylabel("GFP (fT)")
            _ax.set_title(f"Run {RUN_IDX}: Grand average GFP (n={n_kept})")
            _ax.legend(fontsize=8)
            _fig_gfp.tight_layout()
            _fig_gfp.savefig(str(_fd / "epochs_gfp.png"), dpi=150, bbox_inches='tight')
            plt.close(_fig_gfp)

            _peak_t = float(_times[np.argmax(_gfp)])
            _topo_t = sorted(set(
                [round(_peak_t, 3)] +
                [round((_t0 + _t1) / 2, 3) for _t0, _t1 in globals().get("ROI_WINDOWS", {}).values()]
            ))
            _topo_t = [t for t in _topo_t if TMIN <= t <= TMAX]
            _fig_topo = _evoked.plot_topomap(times=_topo_t, show=False, time_unit='s')
            _fig_topo.suptitle(f"Run {RUN_IDX}: Topomap", fontsize=10)
            _fig_topo.savefig(str(_fd / "epochs_topomap.png"), dpi=150, bbox_inches='tight')
            plt.close(_fig_topo)

            log(f"[run {RUN_IDX}] Saved epochs QC figures -> {_fd.name}/")
    except Exception as _e:
        log(f"[run {RUN_IDX}] Epochs figures skipped: {_e}")

    #Remaining QC (annotations, PSD of clean, general info) -------------
    try:
        bad_annots = [ann for ann in rf.annotations if 'bad' in ann['description'].lower()]
        total_bad_time = sum(ann['duration'] for ann in bad_annots)
        total_duration = rf.times[-1] - rf.times[0]
        qc_collector["annotations_qc"] = {
            "total_duration_sec":       float(total_duration),
            "bad_segments_count":       len(bad_annots),
            "total_bad_duration_sec":   float(total_bad_time),
            "fraction_bad":             float(total_bad_time / total_duration) if total_duration > 0 else 0.0,
            "bad_descriptions":         list(set([ann['description'] for ann in bad_annots]))
        }

        try:
            spectrum = raw_clean.compute_psd(fmin=1, fmax=40, n_jobs=1, verbose='ERROR')
            freqs    = spectrum.freqs
            psd      = spectrum.get_data()
            freq_indices = range(0, len(freqs), max(1, len(freqs) // 10))
            psd_mean_by_freq = {f"f{int(freqs[i])}Hz": float(np.mean(psd[:, i]))
                                for i in freq_indices if i < len(freqs)}
            qc_collector["psd_qc"] = {"psd_summary": psd_mean_by_freq, "n_freqs": len(freqs)}
            log(f"[run {RUN_IDX}] PSD computed: {len(freqs)} frequencies")
            try:
                import matplotlib.pyplot as plt
                _fd = run_dir / "figures"; _fd.mkdir(exist_ok=True)
                _pf_clean = spectrum.plot(show=False)
                _pf_clean.suptitle(f"Run {RUN_IDX}: PSD clean (post-ICA)", fontsize=10)
                _pf_clean.savefig(str(_fd / "psd_clean.png"), dpi=150, bbox_inches='tight')
                plt.close(_pf_clean)
            except Exception as _e:
                log(f"[run {RUN_IDX}] clean PSD figure skipped: {_e}")
        except Exception as e:
            log(f"[run {RUN_IDX}] PSD skipped: {e}")
            qc_collector["psd_qc"] = {"error": str(e)}

        qc_collector["general"] = {
            "run_idx":        int(RUN_IDX),
            "raw_file":       str(RUN_FILE),
            "n_channels":     int(raw.info['nchan']),
            "n_bad_channels": len(raw.info['bads']),
            "bad_channels":   list(raw.info['bads']),
            "sfreq":          float(raw.info['sfreq']),
            "do_sss":         bool(DO_SSS),
            "hp_hz":          float(HP),
            "lp_hz":          float(LP),
            "notch_freqs":    list(map(float, NOTCH)) if NOTCH else [],
        }

        run_qc_path = run_dir / "run_qc_summary.json"
        run_qc_path.write_text(json.dumps(qc_collector, indent=2), encoding="utf-8")
        log(f"[run {RUN_IDX}] Saved comprehensive QC: {run_qc_path.name}")

    except Exception as e:
        log(f"[run {RUN_IDX}] Error in QC collection: {e}")


need_er_after_run1 = (ER_RAW_PATH and ALIGN_EMPTY_ROOM_TO_RUN1
                      and not (DERIV_DIR / f"{SESSION_LABEL}_empty_room" / "raw_er_sss.fif").exists())

if ER_RAW_PATH and not need_er_after_run1:
    process_empty_room()

for run_idx in range(START_RUN_IDX, END_RUN_IDX + 1):
    process_run(run_idx)
    if run_idx == 1 and need_er_after_run1:
        process_empty_room()

log(f"All runs completed (runs {START_RUN_IDX}-{END_RUN_IDX})!")


In [ ]:
if 'log' not in dir():
    def log(*args): print(">>", *args)

import mne
import numpy as np
import pandas as pd

combined_out = DERIV_DIR / f"epochs_combined_{SESSION_LABEL}-epo.fif"

epoch_files = sorted(
    DERIV_DIR.glob("run*/epochs-epo.fif"),
    key=lambda p: int(p.parent.name.replace("run", ""))
)

if not epoch_files:
    raise RuntimeError(f"No per-run epochs-epo.fif found in {DERIV_DIR}")

log(f"[combine] Found {len(epoch_files)} run epoch files")

all_epochs = []
for ep_path in epoch_files:
    run_num = int(ep_path.parent.name.replace("run", ""))
    ep = mne.read_epochs(str(ep_path), preload=True, verbose="ERROR")

    # Tag each epoch with its run number
    ep.metadata = ep.metadata.copy() if ep.metadata is not None else pd.DataFrame(index=range(len(ep)))
    ep.metadata["run"] = run_num

    log(f"  run{run_num:02d}: {len(ep)} epochs, {ep.info['nchan']} channels")
    all_epochs.append(ep)

# Equalize channels across runs (handles runs with different bad-channel counts)
ch_sets = [set(ep.ch_names) for ep in all_epochs]
common_chs = sorted(ch_sets[0].intersection(*ch_sets[1:]), key=lambda c: all_epochs[0].ch_names.index(c))
n_total = max(len(ep.ch_names) for ep in all_epochs)
if len(common_chs) < n_total:
    log(f"[combine] WARNING: runs have unequal channels - keeping {len(common_chs)}/{n_total} common channels")

# Always pick to the common channel set to guarantee matching info['nchan']
all_epochs = [ep.pick(common_chs) for ep in all_epochs]

epochs_combined = mne.concatenate_epochs(all_epochs, add_offset=True)

# Add global trial index
epochs_combined.metadata = epochs_combined.metadata.reset_index(drop=True)
epochs_combined.metadata["trial_global"] = np.arange(len(epochs_combined))

epochs_combined.save(str(combined_out), overwrite=True)

log(f"[combine] Saved: {combined_out.name}")
log(f"[combine] Total epochs: {len(epochs_combined)}")
log(f"[combine] Runs combined: {sorted(epochs_combined.metadata['run'].unique())}")
log(f"[combine] Time: {epochs_combined.tmin:.3f} .. {epochs_combined.tmax:.3f} s")


## 5. GROUP QC REPORT (All Subjects)

QC Report for all subjects

In [ ]:
import logging
from datetime import datetime
from typing import List, Dict, Tuple
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

MEG_ROOT = Path(r"E:\Develop\MEG")
QC_REPORT_DIR = MEG_ROOT / "QC_REPORT_GROUP"
QC_REPORT_DIR.mkdir(parents=True, exist_ok=True)

MIN_EPOCHS_PER_RUN = 10
MIN_TOTAL_EPOCHS_PER_SUBJECT = 30

LOG_FILE = QC_REPORT_DIR / "qc_errors.log"
logger = logging.getLogger("GroupQC")
logger.handlers.clear()
logger.setLevel(logging.DEBUG)
fh = logging.FileHandler(LOG_FILE, encoding='utf-8')
fh.setFormatter(logging.Formatter('[%(asctime)s] %(levelname)s: %(message)s'))
logger.addHandler(fh)

console = logging.StreamHandler()
console.setLevel(logging.INFO)
console.setFormatter(logging.Formatter('[%(levelname)s] %(message)s'))
logger.addHandler(console)


def find_all_subjects(root_dir: Path) -> List[Path]:
    subjects = []
    for subject_dir in root_dir.iterdir():
        if not subject_dir.is_dir() or subject_dir.name.startswith('QC_'):
            continue

        deriv1 = subject_dir / "derivatives"
        deriv2 = subject_dir / "S" / "derivatives"
        deriv3 = subject_dir / "C" / "derivatives"

        for deriv_path in [deriv1, deriv2, deriv3]:
            if deriv_path.exists() and list(deriv_path.glob("*/run_qc_summary.json")):
                subjects.append(deriv_path)

    return sorted(subjects)


def extract_subject_id(deriv_path: Path) -> str:
    parts = deriv_path.parts
    for i, part in enumerate(parts):
        if part == "derivatives" and i > 0:
            candidate = parts[i - 2] if i >= 2 and parts[i-1] in ["S", "C"] else parts[i - 1]
            return candidate
    return deriv_path.parent.parent.name


def find_run_qc_files(deriv_path: Path) -> Dict[int, Path]:
    qc_by_run = {}

    for run_dir in deriv_path.glob("run*"):
        if not run_dir.is_dir():
            continue

        run_name = run_dir.name
        try:
            run_idx = int(run_name.replace("run", ""))
        except ValueError:
            continue

        qc_files = list(run_dir.glob("run_qc_summary.json"))
        if qc_files:
            qc_by_run[run_idx] = qc_files[0]

    return qc_by_run


def collect_subject_qc(deriv_path: Path) -> pd.DataFrame:
    subject_id = extract_subject_id(deriv_path)
    qc_by_run = find_run_qc_files(deriv_path)

    if not qc_by_run:
        logger.warning(f"⚠ {subject_id}: No QC files found")
        return pd.DataFrame()

    rows = []
    for run_idx, qc_file in sorted(qc_by_run.items()):
        try:
            with open(qc_file, 'r', encoding='utf-8') as f:
                qc_data = json.load(f)

            row = {
                "subject": subject_id,
                "run": run_idx,
            }

            if "epochs_qc" in qc_data:
                row.update({f"epochs_{k}": v for k, v in qc_data["epochs_qc"].items()})

            if "ica_qc" in qc_data:
                row.update({f"ica_{k}": v for k, v in qc_data["ica_qc"].items()})

            if "annotations_qc" in qc_data:
                row.update({f"annot_{k}": v for k, v in qc_data["annotations_qc"].items()})

            if "psd_qc" in qc_data:
                row.update({f"psd_{k}": v for k, v in qc_data["psd_qc"].items()})

            rows.append(row)
            logger.info(f"✓ {subject_id} run{run_idx}: {qc_data['epochs_qc'].get('n_epochs_kept', 'N/A')} epochs")

        except Exception as e:
            logger.warning(f"⚠ {subject_id} run{run_idx}: {e}")

    return pd.DataFrame(rows)

logger.info("="*70)
logger.info("STARTING GROUP QC SCAN (Reading from run_qc_summary.json)")
logger.info("="*70)

subjects = find_all_subjects(MEG_ROOT)
logger.info(f"Found {len(subjects)} subjects\n")

all_qc_data = []
for subject_deriv in subjects:
    logger.info(f"Processing: {subject_deriv.parent.parent.name}...")
    try:
        subject_qc = collect_subject_qc(subject_deriv)
        if not subject_qc.empty:
            all_qc_data.append(subject_qc)
    except Exception as e:
        logger.error(f"Failed: {e}")

if all_qc_data:
    qc_df = pd.concat(all_qc_data, ignore_index=True)

    qc_df.to_csv(QC_REPORT_DIR / "qc_subjects.tsv", sep='\t', index=False, encoding='utf-8')
    logger.info(f"\n✓ Saved: qc_subjects.tsv ({len(qc_df)} rows)")

    print("\n" + "="*70)
    print("SUMMARY STATISTICS")
    print("="*70)
    summary_cols = ['subject', 'run', 'ica_n_excluded', 'annot_total_bad_duration_sec', 'annot_fraction_bad']
    available_cols = [col for col in summary_cols if col in qc_df.columns]
    print(qc_df[available_cols].to_string())
else:
    logger.error("No QC data collected!")
    qc_df = None


In [ ]:
# Plots and reports


if qc_df is not None and len(qc_df) > 0:
    figures_dir = QC_REPORT_DIR / "figures"
    figures_dir.mkdir(exist_ok=True)

    unique_subjects = sorted(qc_df['subject'].unique())
    subject_to_num = {subject: i+1 for i, subject in enumerate(unique_subjects)}
    qc_df['subject_num'] = qc_df['subject'].map(subject_to_num)

    qc_display = qc_df.copy()
    qc_display['subject'] = qc_display['subject_num']
    qc_display = qc_display.drop('subject_num', axis=1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    if 'annot_fraction_bad' in qc_df.columns:
        sns.boxplot(data=qc_df, x='subject_num', y='annot_fraction_bad', ax=axes[0])
        axes[0].set_ylabel('Fraction of BAD Time')
        axes[0].set_xlabel('Subject ID')
        axes[0].set_title('Data Quality: BAD Annotations')
        axes[0].tick_params(axis='x', rotation=0)

    if 'ica_n_excluded' in qc_df.columns:
        sns.boxplot(data=qc_df, x='subject_num', y='ica_n_excluded', ax=axes[1])
        axes[1].set_ylabel('Number of Components Excluded')
        axes[1].set_xlabel('Subject ID')
        axes[1].set_title('ICA Aggressiveness')
        axes[1].tick_params(axis='x', rotation=0)

    plt.tight_layout()
    plt.savefig(figures_dir / "01_data_quality.pdf", dpi=300, bbox_inches='tight')
    plt.savefig(figures_dir / "01_data_quality.png", dpi=150, bbox_inches='tight')
    plt.close()
    logger.info("Saved: data quality figure")

    # BAD annotations count
    fig, ax = plt.subplots(figsize=(10, 6))
    data_bad = qc_df.dropna(subset=['annot_bad_segments_count']).copy()
    if len(data_bad) > 0 and 'annot_bad_segments_count' in data_bad.columns:
        sns.barplot(data=data_bad, x='subject_num', y='annot_bad_segments_count', ax=ax)
        ax.set_ylabel('Number of BAD Segments')
        ax.set_xlabel('Subject ID')
        ax.set_title('Data Quality: BAD Segment Count')
        ax.tick_params(axis='x', rotation=0)
        plt.tight_layout()
        plt.savefig(figures_dir / "02_bad_segments.pdf", dpi=300, bbox_inches='tight')
        plt.savefig(figures_dir / "02_bad_segments.png", dpi=150, bbox_inches='tight')
        plt.close()
        logger.info("✓ Saved: BAD segments figure")

    # ICA components
    fig, ax = plt.subplots(figsize=(10, 6))
    if 'ica_n_components' in qc_df.columns:
        sns.barplot(data=qc_df, x='subject_num', y='ica_n_components', ax=ax, color='steelblue')
        ax.set_ylabel('Number of ICA Components')
        ax.set_xlabel('Subject ID')
        ax.set_title('ICA: Number of Components Extracted')
        ax.tick_params(axis='x', rotation=0)
        plt.tight_layout()
        plt.savefig(figures_dir / "03_ica_components.pdf", dpi=300, bbox_inches='tight')
        plt.savefig(figures_dir / "03_ica_components.png", dpi=150, bbox_inches='tight')
        plt.close()
        logger.info("✓ Saved: ICA components figure")


    # SUMMARY JSON

    summary = {
        "generation_date": datetime.now().isoformat(),
        "n_subjects": int(n_subjects),
        "n_runs": int(n_runs),
        "bad_time_fraction": {
            "mean": float(bad_frac_mean),
            "std": float(qc_df['annot_fraction_bad'].std()) if 'annot_fraction_bad' in qc_df.columns else 0.0,
        },
        "ica_components_excluded": {
            "mean": float(ica_excl_mean),
            "std": float(qc_df['ica_n_excluded'].std()) if 'ica_n_excluded' in qc_df.columns else 0.0,
        }
    }

    (QC_REPORT_DIR / "qc_summary.json").write_text(json.dumps(summary, indent=2), encoding='utf-8')
    logger.info("✓ Saved: qc_summary.json")

    logger.info("\n" + "="*70)
    logger.info("GROUP QC REPORT COMPLETE")
    logger.info(f"Output directory: {QC_REPORT_DIR}")
    logger.info("="*70)

In [ ]:
#Exclusions, motion, HTML Report


def extract_headpos_qc(run_dir: Path) -> Dict:
    qc = {
        "motion_median_mm": np.nan,
        "motion_max_mm": np.nan,
        "motion_p95_mm": np.nan,
    }

    hp_files = list(run_dir.glob("*headpos*.json"))
    if not hp_files:
        return qc

    try:
        with open(hp_files[0], 'r', encoding='utf-8') as f:
            data = json.load(f)
        qc["motion_median_mm"] = data.get("movement_mm_p50", np.nan)
        qc["motion_max_mm"] = data.get("movement_mm_max", np.nan)
        qc["motion_p95_mm"] = data.get("movement_mm_p95", np.nan)
    except Exception as e:
        pass

    return qc


def generate_exclusion_report(qc_df: pd.DataFrame,
                              min_bad_fraction: float = 0.05) -> Tuple[pd.DataFrame, str]:

    exclusions = []

    bad_runs = qc_df[qc_df['annot_fraction_bad'] > min_bad_fraction]
    for _, row in bad_runs.iterrows():
        exclusions.append({
            'subject_id': int(row['subject_num']) if 'subject_num' in row else row['subject'],
            'run': int(row['run']),
            'reason': f"High BAD fraction ({row['annot_fraction_bad']:.2%} > {min_bad_fraction:.2%})",
            'bad_fraction': float(row['annot_fraction_bad']),
        })

    excl_df = pd.DataFrame(exclusions)

    n_excluded_runs = len(bad_runs)
    n_total_runs = len(qc_df)
    n_included_runs = n_total_runs - n_excluded_runs

    text = f"""
DATA QUALITY SUMMARY

Quality Criteria:
  • Per-run: maximum {min_bad_fraction:.1%} BAD time fraction

Results:
  • Total runs: {n_total_runs}
  • Excluded runs (low QC): {n_excluded_runs}
  • INCLUDED runs (for analysis): {n_included_runs}

Detailed quality metrics:
- Mean BAD fraction: {qc_df['annot_fraction_bad'].mean():.2%}
- Max BAD fraction: {qc_df['annot_fraction_bad'].max():.2%}
- Mean ICA components excluded: {qc_df['ica_n_excluded'].mean():.1f}

Excluded runs:
{excl_df.to_string(index=False) if len(excl_df) > 0 else "  (none)"}
    """

    return excl_df, text.strip()


if qc_df is not None and len(qc_df) > 0:

    # EXCLUSION REPORT

    excl_df, excl_text = generate_exclusion_report(qc_df, min_bad_fraction=0.05)

    (QC_REPORT_DIR / "data_quality_report.txt").write_text(excl_text, encoding='utf-8')
    excl_df.to_csv(QC_REPORT_DIR / "excluded_runs.csv", index=False, encoding='utf-8')
    logger.info("✓ Saved: data quality report")

    # SAVE UPDATED QC TABLE
    qc_df.to_csv(QC_REPORT_DIR / "qc_subjects_full.tsv", sep='\t', index=False, encoding='utf-8')
    logger.info("✓ Saved: qc_subjects_full.tsv")


    # HTML REPORT

    html_content = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset='utf-8'>
    <title>MEG Preprocessing QC Report</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 20px 40px; background: #f9f9f9; }}
        .container {{ max-width: 1200px; margin: 0 auto; background: white; padding: 20px; border-radius: 8px; }}
        h1, h2 {{ color: #333; border-bottom: 2px solid #0066cc; padding-bottom: 10px; }}
        h1 {{ font-size: 28px; }}
        h2 {{ font-size: 20px; margin-top: 30px; }}
        table {{ border-collapse: collapse; width: 100%; margin: 15px 0; }}
        th, td {{ border: 1px solid #ddd; padding: 10px; text-align: left; }}
        th {{ background-color: #0066cc; color: white; }}
        tr:nth-child(even) {{ background-color: #f2f2f2; }}
        pre {{
            background-color: #f5f5f5;
            padding: 15px;
            border-left: 3px solid #0066cc;
            overflow-x: auto;
            border-radius: 4px;
        }}
        img {{ max-width: 100%; height: auto; margin: 20px 0; border: 1px solid #ddd; border-radius: 4px; }}
        .section {{ margin: 30px 0; }}
        .stats-box {{ background: #e6f2ff; padding: 15px; border-radius: 4px; border-left: 4px solid #0066cc; }}
        .warning {{ background: #fff3cd; padding: 15px; border-radius: 4px; border-left: 4px solid #ff9800; }}
        .success {{ background: #e8f5e9; padding: 15px; border-radius: 4px; border-left: 4px solid #4caf50; }}
        footer {{ text-align: center; color: #999; margin-top: 40px; padding-top: 20px; border-top: 1px solid #ddd; }}
    </style>
</head>
<body>
<div class="container">
    <h1>MEG Preprocessing Quality Control Report</h1>
    <p><strong>Generated:</strong> {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>

    <div class="section stats-box">
        <h3>Summary Statistics</h3>
        <ul>
            <li><strong>Total Subjects:</strong> {qc_df['subject'].nunique()}</li>
            <li><strong>Total Runs:</strong> {len(qc_df)}</li>
            <li><strong>Mean BAD Time Fraction:</strong> {qc_df['annot_fraction_bad'].mean():.2%}</li>
            <li><strong>Mean ICA Components Excluded:</strong> {qc_df['ica_n_excluded'].mean():.1f}</li>
        </ul>
    </div>

    <div class="section">
        <h2>Methods: Preprocessing & QC</h2>
        <pre>{methods_text}</pre>
    </div>

    <div class="section">
        <h2>Data Quality Summary</h2>
        <pre>{excl_text}</pre>
    </div>

    <div class="section">
        <h2>Detailed QC Metrics by Subject and Run</h2>
        {qc_display.to_html(index=False, float_format=lambda x: f'{x:.4f}' if isinstance(x, (float, np.number)) else str(x))}
    </div>

    <div class="section">
        <h2>Quality Control Figures</h2>
"""

    for fig_file in sorted(figures_dir.glob("*.png")):
        fig_rel = fig_file.relative_to(QC_REPORT_DIR)
        html_content += f'<h3>{fig_file.stem.replace("_", " ").title()}</h3>\n'
        html_content += f'<img src="figures/{fig_file.name}" alt="{fig_file.stem}">\n'

    html_content += """
    </div>

    <footer>
        <p>Report generated by MEG Preprocessing Pipeline</p>
        <p>All preprocessing parameters and QC code available at repository</p>
    </footer>
</div>
</body>
</html>
    """

    (QC_REPORT_DIR / "qc_report.html").write_text(html_content, encoding='utf-8')
    logger.info("✓ Saved: qc_report.html")

    print("\n" + "="*70)
    print("FINAL OUTPUT FILES:")
    print("="*70)
    for f in sorted(QC_REPORT_DIR.glob("*.[tj]*")):
        print(f"  ✓ {f.name}")
    for f in sorted((QC_REPORT_DIR / "figures").glob("*")):
        print(f"  ✓ figures/{f.name}")
    print(f"\nOPEN IN BROWSER: {QC_REPORT_DIR / 'qc_report.html'}")
    print("="*70)